In [1]:
import os
print(os.getcwd())

import sys
sys.path.append('../scripts')
from importlib import import_module
ca_module = import_module('08_context_assembler')

assembler = ca_module.ContextAssembler()

# 拿几个真实chunk测token估算准不准
test_texts = [
    "Metformin is widely used as first-line therapy for type 2 diabetes.",
    "二甲双胍是2型糖尿病一线用药，通过抑制肝糖异生发挥降糖作用。",
    "Metformin (二甲双胍) reduces cardiovascular risk in diabetic patients through AMPK activation pathways."
]

for t in test_texts:
    n_tokens = assembler.estimate_tokens(t)
    print(f"[{n_tokens} tokens] {t[:50]}...")


/Users/siqinling/medrag_project/notebooks


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[16 tokens] Metformin is widely used as first-line therapy for...
[22 tokens] 二甲双胍是2型糖尿病一线用药，通过抑制肝糖异生发挥降糖作用。...
[21 tokens] Metformin (二甲双胍) reduces cardiovascular risk in di...


In [2]:
import sys
sys.path.append('../scripts')
from importlib import import_module

pipeline_module = import_module('07_retrieval_pipeline')
pipeline = pipeline_module.MedRAGPipeline(
    chunks_path="../data/processed/chunks.parquet",
    chroma_db_path="../data/processed/chroma_db"
)


result = pipeline.run("What is the effect of metformin on cardiovascular disease?")


print(type(result))
print(result.keys() if isinstance(result, dict) else "不是dict")


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 848.04it/s]
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/y5/4900bss12yg8nhh_4m3wz5q00000gn/T/jieba.cache


构建BM25索引...


Loading model cost 0.281 seconds.
Prefix dict has been built successfully.


初始化 Reranker...
加载reranker模型...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 8009.57it/s]


<class 'list'>
不是dict


In [3]:
print(len(result))
print(type(result[0]))
print(result[0])

10
<class 'dict'>
{'chunk_id': 'PMC2946277_chunk0', 'text': 'Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention. Materials and methods All patients aged 30 years or older receiving glucose-lowering drugs (GLDs) and admitted with myocardial infarction (MI) not treated with emergent percutaneous coronary intervention in Denmark during 1997-2006 were identified by individual-level lin

In [4]:
converted = assembler._convert_to_chunks(result)
print(len(converted))
print(converted[0])


10
DocumentChunk(text='Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention. Materials and methods All patients aged 30 years or older receiving glucose-lowering drugs (GLDs) and admitted with myocardial infarction (MI) not treated with emergent percutaneous coronary intervention in Denmark during 1997-2006 were identified by individual-level linkage of nationwide registries of hospi

In [5]:
deduped = assembler._deduplicate(converted)
print(f"去重前: {len(converted)}, 去重后: {len(deduped)}")

# 顺手看看有没有明显相似的chunk被识别出来
for i in range(len(converted)):
    for j in range(i+1, len(converted)):
        sim = assembler._jaccard_similarity(converted[i].text, converted[j].text)
        if sim > 0.3:
            print(f"chunk {i} vs {j}: 相似度 {sim:.2f}")


去重前: 10, 去重后: 10


In [6]:
test_chunks = [
    ca_module.DocumentChunk(text="A1", metadata={}, relevance_score=0.9, source="PMC001", chunk_id="PMC001_chunk0"),
    ca_module.DocumentChunk(text="A2", metadata={}, relevance_score=0.85, source="PMC001", chunk_id="PMC001_chunk1"),
    ca_module.DocumentChunk(text="A3", metadata={}, relevance_score=0.8, source="PMC001", chunk_id="PMC001_chunk2"),
    ca_module.DocumentChunk(text="B1", metadata={}, relevance_score=0.75, source="PMC002", chunk_id="PMC002_chunk0"),
    ca_module.DocumentChunk(text="C1", metadata={}, relevance_score=0.7, source="PMC003", chunk_id="PMC003_chunk0"),
]

ranked = assembler._diversify_and_rank(test_chunks)
for c in ranked:
    print(f"{c.chunk_id}: {c.relevance_score}")


PMC001_chunk0: 0.9
PMC002_chunk0: 0.75
PMC003_chunk0: 0.7
PMC001_chunk1: 0.85
PMC001_chunk2: 0.8


In [7]:
final_result = assembler.assemble_context(result)

print(final_result["metadata"])
print("---")
print(final_result["context_text"][:1000])


{'total_chunks_retrieved': 10, 'unique_chunks_after_dedup': 10, 'chunks_selected': 8, 'estimated_tokens': 2985, 'chunk_sources': {'unique_sources': 8, 'source_distribution': {'PMC2946277': 1, 'PMC2546413': 1, 'PMC2566605': 1, 'PMC2991324': 1, 'PMC1974811': 1, 'PMC2664796': 1, 'PMC2940872': 1, 'PMC2797799': 1}}}
---
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated

In [8]:
small_assembler = ca_module.ContextAssembler(max_context_tokens=200)
small_result = small_assembler.assemble_context(result)
print(small_result["metadata"])
print(small_result["context_text"])


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
{'total_chunks_retrieved': 10, 'unique_chunks_after_dedup': 10, 'chunks_selected': 1, 'estimated_tokens': 144, 'chunk_sources': {'unique_sources': 1, 'source_distribution': {'PMC2946277': 1}}}
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention

In [9]:
import sys
sys.path.append('../scripts')
from importlib import import_module

pt_module = import_module('09_prompt_templates')

stage = pt_module.EVIDENCE_EVALUATOR
filled_user_prompt = stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500]
)
print(stage.system_prompt)
print("---")
print(filled_user_prompt)


You are a medical evidence evaluator. Your task is to critically assess a set of retrieved literature excerpts in relation to a clinical or biomedical question.

For each excerpt, evaluate:
1. Relevance: Does it directly address the question, or only tangentially related?
2. Evidence strength: What type of study is it (e.g. RCT, cohort study, case report, review)? Larger, controlled, and more recent studies generally carry more weight.
3. Consistency: Does it agree or conflict with other excerpts provided?

Do not answer the question yet. Only produce a structured evaluation of the evidence. Be concise and avoid restating the full text of each excerpt — reference them by source ID.
---
Question: What is the effect of metformin on cardiovascular disease?

Retrieved evidence:
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emer

In [10]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

ag_stage = pt_module.ANSWER_GENERATOR
filled = ag_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500],
    evidence_evaluation="[占位] Source PMC2946277: relevance high, cohort study, moderate evidence strength."
)
print(ag_stage.system_prompt)
print("---")
print(filled)


You are a medical literature assistant. Your task is to draft an evidence-based answer to a clinical or biomedical question, using only the retrieved literature excerpts and the evidence evaluation provided.

Rules:
1. Base your answer strictly on the provided evidence. Do not introduce facts, mechanisms, or statistics that are not present in the excerpts.
2. Prioritize evidence rated as high relevance and strong evidence quality in the evaluation. Give less weight to low-relevance or weak-quality sources, and mention if evidence is limited or mixed.
3. Cite each claim with its source ID (e.g. [PMC2946277]) immediately after the statement it supports.
4. If the evidence is insufficient or conflicting on some aspect of the question, say so explicitly rather than filling the gap with assumptions.
5. Write in clear, precise scientific English suitable for a clinical audience.
---
Question: What is the effect of metformin on cardiovascular disease?

Retrieved evidence:
[来源: PMC2946277 | 期刊

In [11]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

cr_stage = pt_module.CRITICAL_REVIEWER
filled = cr_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500],
    draft_answer="[占位] Metformin reduces cardiovascular mortality in diabetic patients [PMC2946277]."
)
print(cr_stage.system_prompt)
print("---")
print(filled)


You are a critical reviewer specializing in evidence-based medicine. Your task is to fact-check a draft answer against the original retrieved evidence, and identify any issues.

Check specifically for:
1. Hallucination: Does the draft state any fact, number, or mechanism that is NOT actually present in the retrieved evidence?
2. Citation accuracy: Does each cited source ID actually support the claim it's attached to?
3. Overreach: Does the draft state a causal conclusion when the underlying evidence only supports a correlational or associative finding (e.g. observational/cohort studies)?
4. Missed uncertainty: Are there points where evidence is weak, limited, or conflicting, but the draft states them with unwarranted confidence?

Do not rewrite the answer yourself. Only produce a list of issues found, each with a brief explanation and, where applicable, a suggested correction. If no issues are found for a category, state that explicitly.
---
Question: What is the effect of metformin on

In [12]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

fa_stage = pt_module.FINAL_ASSEMBLER
filled = fa_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    draft_answer="[占位] Metformin reduces cardiovascular mortality in diabetic patients [PMC2946277].",
    review_issues="[占位] Overreach: the cited study is a cohort study, so 'reduces' should be softened to 'associated with lower' unless a causal design is confirmed."
)
print(fa_stage.system_prompt)
print("---")
print(filled)


You are a medical literature assistant preparing a final answer for a Chinese-speaking user. You will be given a draft answer (in English) and a list of issues found during critical review.

Your task:
1. Revise the draft to address every issue raised in the review — remove any unsupported claims, soften overreaching causal language into correlational language where appropriate, and add explicit notes of uncertainty where the review flagged weak or conflicting evidence.
2. Keep all source citations (e.g. [PMC2946277]) attached to their corresponding claims.
3. Translate the final, corrected answer into clear, natural Chinese suitable for a medical student or researcher. Do not simply translate the flawed draft — translate the corrected version.
4. If the evidence is genuinely insufficient to answer part of the question, state this clearly in Chinese rather than omitting it silently.

Output only the final Chinese answer. Do not include your revision process or the English draft.
---
Qu

In [13]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

for stage_key, stage in pt_module.PROMPT_STAGES.items():
    print(f"{stage_key} -> {stage.name} (temperature={stage.temperature}, max_tokens={stage.max_tokens})")

evidence_evaluator -> 证据评估器 (temperature=0.2, max_tokens=800)
answer_generator -> 答案生成器 (temperature=0.3, max_tokens=1000)
critical_reviewer -> 批判性审查器 (temperature=0.2, max_tokens=800)
final_assembler -> 最终组装器 (temperature=0.3, max_tokens=1200)


In [1]:
import sys
sys.path.insert(0, "../scripts")

from importlib import import_module

retrieval_module = import_module("07_retrieval_pipeline")

MedRAGPipeline = retrieval_module.MedRAGPipeline

retrieval_pipeline = MedRAGPipeline(
    chunks_path="../data/processed/chunks.parquet",
    chroma_db_path="../data/processed/chroma_db",
    reranker_model_path="../bge-reranker-base-cache",
)


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1179.78it/s]
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/y5/4900bss12yg8nhh_4m3wz5q00000gn/T/jieba.cache


构建BM25索引...


Loading model cost 0.432 seconds.
Prefix dict has been built successfully.


初始化 Reranker...
加载reranker模型...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 8853.48it/s]


In [2]:
test_query = "What is the effect of metformin on cardiovascular disease?"

retrieved_docs = retrieval_pipeline.run(test_query, top_k_final=5)
print(f"检索到 {len(retrieved_docs)} 条结果")
for d in retrieved_docs:
    print(d['rank'], d['chunk_id'], round(d['final_score'], 3), d['text'][:50])


检索到 5 条结果
1 PMC2946277_chunk0 0.303 Effects of oral glucose-lowering drugs on long ter
2 PMC2546413_chunk0 0.23 Fatal hemolytic anemia associated with metformin: 
3 PMC2566605_chunk0 0.202 Effect of Adjunct Metformin Treatment in Patients 
4 PMC2991324 0.155 Long-term effect of metformin on blood glucose con
5 PMC1974811_chunk0 0.128 Rosiglitazone RECORD study: glucose control outcom


In [3]:
import json

with open("../data/processed/test_retrieved_docs.json", "w", encoding="utf-8") as f:
    json.dump(retrieved_docs, f, ensure_ascii=False, indent=2)
print("已保存")


已保存


In [4]:
import sys, json
sys.path.insert(0, "../scripts")

from importlib import import_module
generation_module = import_module("11_generation_pipeline")
MedicalGenerationPipeline = generation_module.MedicalGenerationPipeline

with open("../data/processed/test_retrieved_docs.json", "r", encoding="utf-8") as f:
    retrieved_docs = json.load(f)

test_query = "What is the effect of metformin on cardiovascular disease?"

generation_pipeline_lite = MedicalGenerationPipeline(
    llm_model_name="deepseek-r1:7b",
    enable_evidence_evaluation=False,
    enable_critical_review=False,
    llm_timeout=600,
)

result = generation_pipeline_lite.run(test_query, retrieved_docs)
print("答案：")
print(result["answer"])
print("\n耗时：", result["generation_metrics"])


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[OK] 已连接Ollama，模型 'deepseek-r1:7b' 可用。
答案：
根据提供的文献和规则，以下是关于甲磺酸他汀对心血管疾病影响的证据支持的回答：

1. **潜在的心血管风险**：来自2010年发表在《Cardiovascular Diabetology》的研究发现，使用甲磺酸他汀的糖尿病患者较其他降糖药物更有可能出现心血管事件，如心肌梗死、心衰竭和死亡。这种增加的风险主要与药物的长期使用相关，并且可能与患者的初始心血管状态有关。

2. **血友病性贫血**：2008年发表在《Journal of Medical Case Reports》的一份案例报告描述了一位因使用甲磺酸他汀治疗糖尿病而发生致命血友病性贫血的患者。这表明甲磺酸他汀可能引发罕见但严重的血友病性贫血，这是一种潜在且严重的副作用。

3. **有利影响**：2008年发表在《PLoS ONE》的研究显示，在1型糖尿病患者中，联合甲磺酸他汀治疗显著降低了HbA1c水平，并且对BMI较轻的患者效果更佳。此外，2010年发表在《Nutrition & Metabolism》的研究显示，甲磺酸他汀在非肥胖型糖尿病患者的长期血糖控制方面表现出良好的效果。

4. **与罗西格列酮的比较**：2007年发表在《Diabetic Medicine》的研究显示，在2型糖尿病患者中，罗西格列酮与联合甲磺酸他汀和 sulfonylurea 的治疗方案在血糖控制方面差异不大。然而，甲磺酸他汀在某些情况下（如BMI较轻的患者）显示出更好的效果。

5. **证据质量**：尽管有研究表明甲磺酸他汀可能增加心血管风险，并且存在血友病性贫血的报道，但其他研究显示其在某些情况下有助于降低血糖控制。然而，由于证据的质量和样本量的问题，结论尚不明确。

综上所述，甲磺酸他汀对心血管疾病的影响存在矛盾的证据。尽管有研究表明其可能增加心血管事件的风险，并且存在血友病性贫血的报道，但其他研究显示其在某些情况下有助于降低血糖控制。因此，结论尚不明确，需要进一步的研究来确定其确切影响。

[来源: PMC2946277] | [来源: PMC2546413] | [来源: PMC

In [5]:
generation_pipeline_full = MedicalGenerationPipeline(
    llm_model_name="deepseek-r1:7b",
    enable_evidence_evaluation=False,
    enable_critical_review=True,
    llm_timeout=600,
)

result_full = generation_pipeline_full.run(test_query, retrieved_docs)
print("最终答案：")
print(result_full["answer"])
print("\n耗时：", result_full["generation_metrics"])


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[OK] 已连接Ollama，模型 'deepseek-r1:7b' 可用。
最终答案：
### 甲磺酸西格列定向对心血管疾病的影响

#### 对心血管疾病的潜在保护作用
1. **非肥胖型2型糖尿病患者的保护效果**：
   - 一项随机对照试验（RCT）研究了108名非肥胖型2型糖尿病患者，结果显示在三年内使用甲磺酸西格列定向显著降低了HbA1c水平。由于HbA1c是长期血糖控制的指标，这一结果可能与降低心血管疾病风险相关，并且这种保护效果主要体现在非肥胖型2型糖尿病患者中[PMC2946277]。

   - 另一项RCT研究了49名1型糖尿病（T1DM）患者的血糖控制问题。尽管该研究并未直接探讨心血管疾病的影响，但其结果表明甲磺酸西格列定向在改善血糖控制方面具有显著效果，并且这一药物的保护作用可能与2型糖尿病相似[PMC2566605]。

#### 可能的心血管风险
1. **血友病性贫血**：
   - 一项病例报告描述了一名非肥胖型2型糖尿病患者的 fatal hemolytic anemia事件，提示临床医生在使用甲磺酸西格列定向时需高度警惕这种罕见但可能严重的副作用。这种情况通常与患者有既往的血液疾病或肾脏疾病相关[PMC2546413]。

#### 总结和建议
- **总体影响**：现有证据表明，甲磺酸西格列定向可能通过改善血糖控制对心血管疾病产生保护作用。然而，在评估其益处时，医生应权衡潜在风险，特别是血友病性贫血增加的风险。
  
- **患者选择**：该药物最适合于2型糖尿病患者的血糖控制不佳且存在其他危险因素的患者，尤其是那些非肥胖型的患者，因为这些患者可能从较低剂量中获益更多[PMC2946277]。

- **临床应用**：在考虑使用甲磺酸西格列定向时，医生应密切监测血友病性贫血等副作用，并权衡其与潜在替代治疗方案的利弊。最终决策应根据患者的具体医疗历史和反应来确定。

#### 结论
甲磺酸西格列定向可能对心血管疾病具有保护作用，尤其是非肥胖型2型糖尿病患者。然而，在使用时医生需谨慎考虑潜在风险并进行密切监测。目前的研究仍需进一步扩展以全面了解其对心血管疾病长期影响的影响。

**

In [6]:
import sys, json
sys.path.insert(0, "../scripts")

from importlib import import_module
generation_module = import_module("11_generation_pipeline")
MedicalGenerationPipeline = generation_module.MedicalGenerationPipeline

with open("../data/processed/test_retrieved_docs.json", "r", encoding="utf-8") as f:
    retrieved_docs = json.load(f)

test_query = "What is the effect of metformin on cardiovascular disease?"

generation_pipeline_full = MedicalGenerationPipeline(
    llm_model_name="deepseek-r1:7b",
    enable_evidence_evaluation=False,
    enable_critical_review=True,
    llm_timeout=600,
)

result_full = generation_pipeline_full.run(test_query, retrieved_docs)
print("最终答案：")
print(result_full["answer"])
print("\n引用校验：", result_full["citation_check"])
print("\n耗时：", result_full["generation_metrics"])


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[OK] 已连接Ollama，模型 'deepseek-r1:7b' 可用。
最终答案：
### 用中文写的最终答案：

#### 摘要
Metformin，一种大麦素类降糖药，在其对心血管疾病（CVD）的影响方面存在混合研究结果。本回答综合五篇文献的研究成果，旨在探讨Metformin对CVD风险和预后的影响。

---

### 长期效果

1. **血糖控制的积极影响**  
   多项研究表明，Metformin显著改善了非肥胖性2型糖尿病患者的血糖控制。例如：
   - 一项回顾性研究发现，在213名非肥胖性T2DM患者中，Metformin在3年内降低了HbA1c水平（[PMC2946277](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2946277)）。
   - 另一项研究显示，在非肥胖性患者中，Metformin可使HbA1c在12个月内较肥胖患者降低约1.2%（[PMC2566605](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2566605)）。

   这些发现表明，Metformin通过改善血糖控制可能降低CVD风险。

2. **非肥胖性患者中的特殊效果**  
   在非肥胖性T2DM患者中，Metformin因其低剂量即可维持良好血糖控制而显示出特殊的疗效优势（[PMC2946277](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2946277)）。

---

### 短期心血管副作用和心血管风险

1. **血红蛋白减少症**  
   一项病例报告指出，Metformin可能引起罕见的血液学并发症——血红蛋白减少症（[PMC2546413](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2546413)）。这种并发症可能导致严重的心血管事件，如急性呼吸窘迫综合征和死亡。因此，在使用Metformin时，医生应高度警觉并采取必要预防措施。

2. **对CVD风险的短期影响**  
   尽管有研究表

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from importlib import import_module

retrieval_module = import_module("07_retrieval_pipeline")
generation_module = import_module("11_generation_pipeline")
test_module = import_module("12_test_generation_pipeline")

MedRAGPipeline = retrieval_module.MedRAGPipeline
MedicalGenerationPipeline = generation_module.MedicalGenerationPipeline
TEST_QUERIES = test_module.TEST_QUERIES
run_batch_test = test_module.run_batch_test

retrieval_pipeline = MedRAGPipeline(
    chunks_path="../data/processed/chunks.parquet",
    chroma_db_path="../data/processed/chroma_db",
    reranker_model_path="../bge-reranker-base-cache",
)

generation_pipeline_lite = MedicalGenerationPipeline(
    llm_model_name="deepseek-r1:7b",
    enable_evidence_evaluation=False,
    enable_critical_review=False,
    llm_timeout=600,
)

run_batch_test(
    pipeline=generation_pipeline_lite,
    queries=TEST_QUERIES,
    retrieval_pipeline=retrieval_pipeline,
    output_path="../reports/generation_test_log_lite.jsonl",
)


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1424.79it/s]
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/y5/4900bss12yg8nhh_4m3wz5q00000gn/T/jieba.cache


构建BM25索引...


Loading model cost 0.273 seconds.
Prefix dict has been built successfully.


...

Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 7718.31it/s]


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[OK] 已连接Ollama，模型 'deepseek-r1:7b' 可用。

========== [1/5] What is the effect of metformin on cardiovascular disease? ==========
  answer长度: 782字符
  stage_success: {'context_assembly': True, 'answer_generator': True}
  幻觉引用: False
  已写入 ../reports/generation_test_log_lite.jsonl

========== [2/5] What are the differences between metformin and sulfonylureas in treating type 2 diabetes? ==========


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x4a5d68c20>
    label = <none> 
    device = <AGXG13GDevice: 0x12a03b600>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x12a03c200>
        label = <none> 
        device = <AGXG13GDevice: 0x12a03b600>
            name = Apple M1 
    retainedReferences = 1


  answer长度: 2096字符
  stage_success: {'context_assembly': True, 'answer_generator': True}
  幻觉引用: False
  已写入 ../reports/generation_test_log_lite.jsonl

========== [3/5] What are the common side effects of metformin? ==========


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x12f2f42e0>
    label = <none> 
    device = <AGXG13GDevice: 0x12a03b600>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x12a03c200>
        label = <none> 
        device = <AGXG13GDevice: 0x12a03b600>
            name = Apple M1 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x4a60dcd30>
    label = <none> 
    device = <AGXG13GDevice: 0x12a03b600>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x12a03c200>
        label = <none> 
        device = <AGXG13GDev

In [ ]:
import json

with open("../reports/generation_test_log_lite.jsonl", "r", encoding="utf-8") as f:
    logs = [json.loads(line) for line in f]

print("=== 第2条 ===")
print(logs[1]["query"])
print(logs[1]["answer"])
print("\n=== 第4条 ===")
print(logs[3]["query"])
print(logs[3]["answer"])
